# Notebook 1: From Local Agent to HTTP Service

In this notebook, you'll transform a local Strands agent into a production-ready HTTP service. We'll cover:

1. Building a basic agent locally
2. Wrapping it in a FastAPI application
3. Adding streaming support with `stream_async`
4. Containerizing with Docker
5. Testing the containerized service

By the end, you'll have a Docker image ready to deploy to any AWS compute service.

> **Which steps do I need?** It depends on which deployment pattern you'll use in Notebook 02:
>
> | Deployment target | Required steps |
> |-------------------|---------------|
> | **Lambda** | Steps 1, 2, 7 |
> | **Fargate** | Steps 1–6 |
> | **All patterns** | Steps 1–7 (run everything) |
>
> If unsure, run all steps — they take a few minutes and give you the full picture.

## Step 1: Install Dependencies

First, install the required packages.

In [ ]:
!pip install strands-agents strands-agents-tools fastapi uvicorn

## Step 2: Build a Local Agent

Let's start with a simple weather agent that uses the `http_request` tool to fetch data from the National Weather Service API. This is the same agent we'll deploy to production.

In [ ]:
from strands import Agent
from strands_tools import http_request

WEATHER_SYSTEM_PROMPT = """You are a weather assistant with HTTP capabilities. You can:

1. Make HTTP requests to the National Weather Service API
2. Process and display weather forecast data
3. Provide weather information for locations in the United States

When retrieving weather information:
1. First get the grid information using https://api.weather.gov/points/{latitude},{longitude}
2. Then use the returned forecast URL to get the actual forecast

When displaying responses:
- Format weather data in a human-readable way
- Highlight important information like temperature, precipitation, and alerts
- Handle errors appropriately
- Don't ask follow-up questions

Always explain the weather conditions clearly and provide context for the forecast.
"""

# Create the agent locally
weather_agent = Agent(
    system_prompt=WEATHER_SYSTEM_PROMPT,
    tools=[http_request],
)

# Test it
response = weather_agent("What is the weather in Seattle? (latitude: 47.6062, longitude: -122.3321)")
print(response)

## Step 3: Wrap in a FastAPI Application

To serve this agent over HTTP, we wrap it in a FastAPI application. This is the pattern used by Fargate, EC2, and EKS deployments.

Key design decisions:
- **POST endpoint** — agents process natural language input, so we use POST with a JSON body
- **Agent-per-request** — each request gets a fresh agent instance to avoid state leakage between users
- **Error handling** — catch exceptions and return proper HTTP error codes

In [ ]:
%%writefile app.py
"""FastAPI application hosting a Strands weather agent."""

from fastapi import FastAPI, HTTPException
from fastapi.responses import PlainTextResponse, StreamingResponse
from pydantic import BaseModel
from strands import Agent, tool
from strands_tools import http_request

app = FastAPI(title="Weather Agent API")

WEATHER_SYSTEM_PROMPT = """You are a weather assistant with HTTP capabilities. You can:

1. Make HTTP requests to the National Weather Service API
2. Process and display weather forecast data
3. Provide weather information for locations in the United States

When retrieving weather information:
1. First get the grid information using https://api.weather.gov/points/{latitude},{longitude}
2. Then use the returned forecast URL to get the actual forecast

When displaying responses:
- Format weather data in a human-readable way
- Highlight important information like temperature, precipitation, and alerts
- Handle errors appropriately
- Don't ask follow-up questions

At the point where tools are done being invoked and a summary can be presented
to the user, invoke the ready_to_summarize tool and then continue with the summary.
"""


class PromptRequest(BaseModel):
    """Request body for agent endpoints."""
    prompt: str


@app.post("/weather")
async def get_weather(request: PromptRequest):
    """Non-streaming endpoint: returns the full response once complete."""
    if not request.prompt:
        raise HTTPException(status_code=400, detail="No prompt provided")

    try:
        # Create a fresh agent per request to avoid state leakage
        agent = Agent(
            system_prompt=WEATHER_SYSTEM_PROMPT,
            tools=[http_request],
        )
        response = agent(request.prompt)
        return PlainTextResponse(content=str(response))
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.post("/weather-streaming")
async def get_weather_streaming(request: PromptRequest):
    """Streaming endpoint: returns tokens as they are generated."""
    if not request.prompt:
        raise HTTPException(status_code=400, detail="No prompt provided")

    async def generate():
        is_summarizing = False

        @tool
        def ready_to_summarize():
            """Signal that tool invocations are complete and summary can begin."""
            nonlocal is_summarizing
            is_summarizing = True
            return "Ok - continue providing the summary!"

        agent = Agent(
            system_prompt=WEATHER_SYSTEM_PROMPT,
            tools=[http_request, ready_to_summarize],
            callback_handler=None,
        )

        async for event in agent.stream_async(request.prompt):
            if not is_summarizing:
                continue
            if "data" in event:
                yield event["data"]

    return StreamingResponse(generate(), media_type="text/plain")


@app.get("/health")
async def health_check():
    """Health check endpoint for load balancers and container orchestrators."""
    return {"status": "healthy"}

### Understanding the Streaming Pattern

The streaming endpoint uses a **boundary tool** pattern:

1. The agent first uses tools (like `http_request`) to gather information — these intermediate steps are hidden from the user
2. When the agent calls `ready_to_summarize`, we flip a flag
3. From that point on, all generated text is streamed to the client

This ensures users only see the final, polished response — not raw API calls and intermediate reasoning.

The `stream_async` method is an async iterator that yields events as the agent processes. Each event with a `"data"` key contains a text chunk.

## Step 4: Test the FastAPI App Locally

We'll start the uvicorn server as a background process directly from the notebook, test the endpoints, then shut it down. No terminal needed.

In [ ]:
import subprocess
import time
import requests

# Start uvicorn in the background
print("Starting FastAPI server...")
server_process = subprocess.Popen(
    ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

# Wait for the server to be ready
for i in range(10):
    time.sleep(2)
    try:
        resp = requests.get("http://localhost:8000/health", timeout=2)
        if resp.status_code == 200:
            print(f"✅ Server is running! Health check: {resp.json()}")
            break
    except requests.exceptions.ConnectionError:
        print(f"   Waiting for server... ({i+1}/10)")
else:
    print("❌ Server failed to start.")
    server_process.terminate()
    print(server_process.stderr.read().decode())

In [ ]:
import requests

# Test the non-streaming /weather endpoint
print("Testing /weather (non-streaming)...")
print("=" * 60)

response = requests.post(
    "http://localhost:8000/weather",
    json={"prompt": "What is the weather in Seattle? (latitude: 47.6062, longitude: -122.3321)"},
    timeout=120,
)

if response.status_code == 200:
    print("✅ Success!")
    print("-" * 60)
    print(response.text[:1500])
else:
    print(f"❌ Error {response.status_code}: {response.text}")

In [ ]:
import requests

# Test the streaming /weather-streaming endpoint
print("Testing /weather-streaming...")
print("=" * 60)

response = requests.post(
    "http://localhost:8000/weather-streaming",
    json={"prompt": "What is the weather in New York? (latitude: 40.7128, longitude: -74.0060)"},
    stream=True,
    timeout=120,
)

if response.status_code == 200:
    print("✅ Streaming response:")
    print("-" * 60)
    for chunk in response.iter_content(chunk_size=None, decode_unicode=True):
        if chunk:
            print(chunk, end="", flush=True)
    print("\n" + "-" * 60)
else:
    print(f"❌ Error {response.status_code}: {response.text}")

In [ ]:
# Shut down the server
server_process.terminate()
server_process.wait()
print("✅ Server stopped.")

## Step 5: Containerize with Docker

To deploy on AWS (Fargate, EKS, EC2), we need a Docker image. Here's a production-ready Dockerfile:

In [ ]:
%%writefile Dockerfile
FROM public.ecr.aws/docker/library/python:3.12-slim

WORKDIR /app

# Install system dependencies
RUN apt-get update && apt-get install -y \
    git \
    && rm -rf /var/lib/apt/lists/*

# Install Python dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy application code
COPY app.py .

# Create a non-root user for security
RUN useradd -m appuser
USER appuser

# Expose the port the app runs on
EXPOSE 8000

# Run with uvicorn
# - workers: 2 for handling concurrent requests
# - host 0.0.0.0: accept connections from outside the container
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "2"]

### Key Dockerfile Decisions

| Decision | Rationale |
|----------|----------|
| `python:3.12-slim` | Small image size, production-ready |
| Non-root user | Security best practice — limits blast radius if container is compromised |
| `--no-cache-dir` | Reduces image size by not caching pip downloads |
| 2 workers | Handles concurrent requests; tune based on CPU/memory allocation |
| ECR public base | Avoids Docker Hub rate limits in CI/CD pipelines |

## Step 6: Build and Test the Container

Now let's build the Docker image and test it. On SageMaker notebook instances (and EC2), the container automatically inherits IAM credentials from the **instance metadata service** — no need to hardcode or pass credentials.

> **How credentials work:**
> - On SageMaker/EC2: The container accesses the instance metadata endpoint (`169.254.169.254`) to get temporary credentials from the attached IAM role.
> - On Fargate: The task role provides credentials via the ECS credential provider.
> - On EKS: The service account provides credentials via IRSA.
>
> You **never** need to hardcode `AWS_ACCESS_KEY_ID` or `AWS_SECRET_ACCESS_KEY`.

In [ ]:
import subprocess
import time
import requests

# Step 6a: Build the Docker image
print("Building Docker image...")
build_result = subprocess.run(
    ["docker", "build", "-t", "strands-weather-agent", "."],
    capture_output=True, text=True
)
if build_result.returncode != 0:
    print(f"Build failed:\n{build_result.stderr}")
else:
    print("✅ Docker image built successfully!")
    # Show image size
    size_result = subprocess.run(
        ["docker", "images", "strands-weather-agent", "--format", "{{.Size}}"],
        capture_output=True, text=True
    )
    print(f"   Image size: {size_result.stdout.strip()}")

In [ ]:
import subprocess
import time
import requests

# Step 6b: Run the container in the background
# On SageMaker, the container inherits credentials from the instance metadata service.
# We use --network=host so the container can reach the metadata endpoint (169.254.169.254).

print("Starting container...")

# Stop any existing container with the same name
subprocess.run(["docker", "rm", "-f", "weather-agent-test"], capture_output=True)

# Run container: use --network=host to inherit IAM role from instance metadata
run_result = subprocess.run(
    [
        "docker", "run", "-d",
        "--name", "weather-agent-test",
        "-p", "8000:8000",
        "-e", "AWS_DEFAULT_REGION=us-east-1",
        "strands-weather-agent",
    ],
    capture_output=True, text=True
)

if run_result.returncode != 0:
    print(f"Failed to start container:\n{run_result.stderr}")
else:
    container_id = run_result.stdout.strip()[:12]
    print(f"✅ Container started: {container_id}")

    # Wait for the server to be ready
    print("Waiting for server to start...")
    for i in range(15):
        time.sleep(2)
        try:
            resp = requests.get("http://localhost:8000/health", timeout=2)
            if resp.status_code == 200:
                print(f"✅ Server is healthy: {resp.json()}")
                break
        except requests.exceptions.ConnectionError:
            print(f"   Attempt {i+1}/15 - waiting...")
    else:
        print("❌ Server did not start in time. Check logs:")
        logs = subprocess.run(
            ["docker", "logs", "weather-agent-test"],
            capture_output=True, text=True
        )
        print(logs.stdout[-2000:] if logs.stdout else logs.stderr[-2000:])

In [ ]:
import requests

# Step 6c: Test the containerized agent
print("Testing the containerized weather agent...")
print("=" * 60)

response = requests.post(
    "http://localhost:8000/weather",
    json={"prompt": "What is the weather in Seattle? (latitude: 47.6062, longitude: -122.3321)"},
    timeout=120,
)

if response.status_code == 200:
    print("✅ Agent responded successfully!")
    print("-" * 60)
    print(response.text[:1000])  # Print first 1000 chars
else:
    print(f"❌ Error {response.status_code}: {response.text}")

In [ ]:
import subprocess

# Step 6d: Cleanup — stop and remove the container
print("Stopping container...")
subprocess.run(["docker", "rm", "-f", "weather-agent-test"], capture_output=True)
print("✅ Container stopped and removed.")

## Step 7: Lambda Handler (Alternative Pattern)

For Lambda deployments, you don't use FastAPI. Instead, you write a handler function that Lambda invokes directly. This is simpler but doesn't support streaming.

In [ ]:
%%writefile agent_handler.py
"""AWS Lambda handler for the Strands weather agent."""

from strands import Agent
from strands_tools import http_request
from typing import Dict, Any

WEATHER_SYSTEM_PROMPT = """You are a weather assistant with HTTP capabilities. You can:

1. Make HTTP requests to the National Weather Service API
2. Process and display weather forecast data
3. Provide weather information for locations in the United States

When retrieving weather information:
1. First get the grid information using https://api.weather.gov/points/{latitude},{longitude}
2. Then use the returned forecast URL to get the actual forecast

When displaying responses:
- Format weather data in a human-readable way
- Highlight important information like temperature, precipitation, and alerts
- Handle errors appropriately
- Convert technical terms to user-friendly language

Always explain the weather conditions clearly and provide context for the forecast.
"""


def handler(event: Dict[str, Any], _context) -> str:
    """Lambda handler function.

    Args:
        event: Lambda event containing a 'prompt' key
        _context: Lambda context (unused)

    Returns:
        Agent response as a string
    """
    weather_agent = Agent(
        system_prompt=WEATHER_SYSTEM_PROMPT,
        tools=[http_request],
    )

    response = weather_agent(event.get("prompt"))
    return str(response)

## Cleanup

Remove Docker images and temporary artifacts created by this notebook.

> **Note:** The files `app.py`, `Dockerfile`, and `agent_handler.py` are intentionally kept — they are required by [Notebook 02](./02_deploy_to_aws.ipynb) for deployment.

In [ ]:
import subprocess
import os
import shutil

# Remove Docker image (the Dockerfile itself is kept for Notebook 02)
subprocess.run(["docker", "rmi", "-f", "strands-weather-agent"], capture_output=True)
print("✅ Removed Docker image")

# Remove __pycache__ if created
if os.path.exists("__pycache__"):
    shutil.rmtree("__pycache__")

print("✅ Cleaned up temporary artifacts")
print("\nℹ️  Kept app.py, Dockerfile, agent_handler.py (needed by Notebook 02)")
print("✅ Notebook 01 cleanup complete!")

## Summary

In this notebook, you've learned how to:

| Step | What you built |
|------|---------------|
| Local agent | A weather agent using `http_request` tool |
| FastAPI wrapper | HTTP endpoints (sync + streaming) |
| Streaming | `stream_async` with boundary tool pattern |
| Docker container | Production-ready image with non-root user |
| Lambda handler | Serverless alternative (no streaming) |

**Next:** In [Notebook 02](./02_deploy_to_aws.ipynb), we'll deploy this to AWS using Lambda and Fargate.